In [314]:
import sympy as sp
import numpy as np
from sympy import symbols, cos, sin, tan, Matrix, pprint
import scipy
from scipy.linalg import schur


In [315]:
# Define state variables and control input
m, g, km, kf = sp.symbols('m g km kf')  # Position, velocity, control
phi, theta, psi = sp.symbols('phi theta psi')
L = sp.symbols('L')
p, q, r = symbols('p q r')
F1, F2, F3, F4 = symbols('F1 F2 F3 F4')
F = Matrix([F1, F2, F3, F4])
I1, I2, I3 = symbols('I1 I2 I3')
I = Matrix([[I1, 0, 0], [0, I2, 0], [0, 0, I3]])
I_inv = I.inv()
px, py, pz, vx, vy, vz, ax, ay, az = symbols('px py pz vx vy vz ax ay az')
u1, u2, u3, u4 = symbols('u1 u2 u3 u4')
u = Matrix([u1, u2, u3, u4])
F1 = u1**2 *kf
F2 = u2**2 *kf
F3 = u3**2 *kf
F4 = u4**2 *kf
M1 = u1**2 *km
M2 = u2**2 *km
M3 = u3**2 *km
M4 = u4**2 *km

R = sp.Matrix([[cos(psi)*cos(theta) - sin(phi)*sin(psi)*sin(theta), -cos(phi)*sin(psi), cos(psi)*sin(theta) + cos(theta)*sin(phi)*sin(psi)],
                 [cos(theta)*sin(psi) + cos(psi)*sin(phi)*sin(theta), cos(phi)*cos(psi), sin(psi)*sin(theta) - cos(psi)*cos(theta)*sin(phi)],
                 [-cos(phi)*sin(theta), sin(phi), cos(phi)*cos(theta)]])

acc_matrix = (1/m) * ((Matrix([0, 0, -m*g]) + R@Matrix([0, 0, F1 + F2 + F3 + F4])))
ang_matrix = I_inv @ (Matrix([[L*(F2-F4)], [L*(F3-F1)], [M1 - M2 + M3 - M4]]) - (Matrix([[0, p, q], [-p , 0, r], [-q, -r, 0]])@(I@Matrix([p, q, r]))))


In [316]:

print(f"acceleration matrix with shape {acc_matrix.shape}")
pprint(acc_matrix)

acceleration matrix with shape (3, 1)
⎡                                       ⎛     2        2        2        2⎞ ⎤
⎢(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))⋅⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠ ⎥
⎢────────────────────────────────────────────────────────────────────────── ⎥
⎢                                    m                                      ⎥
⎢                                                                           ⎥
⎢                                        ⎛     2        2        2        2⎞⎥
⎢(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))⋅⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠⎥
⎢───────────────────────────────────────────────────────────────────────────⎥
⎢                                     m                                     ⎥
⎢                                                                           ⎥
⎢                ⎛     2        2        2        2⎞                        ⎥
⎢         -g⋅m + ⎝kf⋅u₁  + kf⋅u₂  + kf⋅u₃  + kf⋅u₄ ⎠⋅cos(φ)⋅cos(θ)          ⎥
⎢         ────────────────

In [317]:
print(f"angular acceleration matrix with shape {acc_matrix.shape}")
pprint(ang_matrix)

angular acceleration matrix with shape (3, 1)
⎡                           ⎛     2        2⎞       ⎤
⎢      -I₂⋅p⋅q - I₃⋅q⋅r + L⋅⎝kf⋅u₂  - kf⋅u₄ ⎠       ⎥
⎢      ──────────────────────────────────────       ⎥
⎢                        I₁                         ⎥
⎢                                                   ⎥
⎢           2       2     ⎛       2        2⎞       ⎥
⎢       I₁⋅p  - I₃⋅r  + L⋅⎝- kf⋅u₁  + kf⋅u₃ ⎠       ⎥
⎢       ─────────────────────────────────────       ⎥
⎢                         I₂                        ⎥
⎢                                                   ⎥
⎢                       2        2        2        2⎥
⎢I₁⋅p⋅q + I₂⋅q⋅r + km⋅u₁  - km⋅u₂  + km⋅u₃  - km⋅u₄ ⎥
⎢───────────────────────────────────────────────────⎥
⎣                         I₃                        ⎦


In [318]:
# This is the Jacobian of the acceleration matrix w.r.t. input u
C = acc_matrix.jacobian(u)
pprint(C)

⎡2⋅kf⋅u₁⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₂⋅(sin(φ)⋅sin(ψ)⋅cos(θ
⎢──────────────────────────────────────────────   ────────────────────────────
⎢                      m                                                m     
⎢                                                                             
⎢2⋅kf⋅u₁⋅(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))  2⋅kf⋅u₂⋅(-sin(φ)⋅cos(ψ)⋅cos(
⎢───────────────────────────────────────────────  ────────────────────────────
⎢                       m                                                m    
⎢                                                                             
⎢             2⋅kf⋅u₁⋅cos(φ)⋅cos(θ)                            2⋅kf⋅u₂⋅cos(φ)⋅
⎢             ─────────────────────                            ───────────────
⎣                       m                                                m    

) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₃⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₄⋅
──────────────────   ──────────────────────────────

In [319]:
# This is the Jacobian of the angular acceleration matrix w.r.t. input u
E = ang_matrix.jacobian(u)
pprint(E)

⎡             2⋅L⋅kf⋅u₂             -2⋅L⋅kf⋅u₄ ⎤
⎢     0       ─────────      0      ───────────⎥
⎢                 I₁                     I₁    ⎥
⎢                                              ⎥
⎢-2⋅L⋅kf⋅u₁              2⋅L⋅kf⋅u₃             ⎥
⎢───────────      0      ─────────       0     ⎥
⎢     I₂                     I₂                ⎥
⎢                                              ⎥
⎢  2⋅km⋅u₁    -2⋅km⋅u₂    2⋅km⋅u₃    -2⋅km⋅u₄  ⎥
⎢  ───────    ─────────   ───────    ───────── ⎥
⎣     I₃          I₃         I₃          I₃    ⎦


In [320]:

# Compute Jacobians for linearization
# A = f.jacobian(X)  # State matrix (df/dX)
# B = f.jacobian(sp.Matrix([u]))  # Input matrix (df/du)

C = acc_matrix.jacobian(u)
D = ang_matrix.jacobian(Matrix([p, q, r]))
E = ang_matrix.jacobian(u)

final_b_matrix = Matrix([
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    C,
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    E
])


final_a_matrix = Matrix([
                [0,0,0,1,0,0,0,0,0,0,0,0],
                [0,0,0,0,1,0,0,0,0,0,0,0],
                [0,0,0,0,0,1,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,0],
                [0,0,0,0,0,0,0,0,0,1,0,0],
                [0,0,0,0,0,0,0,0,0,1,0,0],
                [0,0,0,0,0,0,0,0,0,0,0,1],
                [0,0,0,0,0,0,0,0,0, *D.row(0)],
                [0,0,0,0,0,0,0,0,0, *D.row(1)],
                [0,0,0,0,0,0,0,0,0, *D.row(2)]])


In [321]:
print(f"final_a_matrix (shape:({final_a_matrix.shape}))")
pprint(final_a_matrix)

final_a_matrix (shape:((12, 12)))
⎡0  0  0  1  0  0  0  0  0    0          0           0    ⎤
⎢                                                         ⎥
⎢0  0  0  0  1  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  1  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    0          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    1          0           0    ⎥
⎢                                                         ⎥
⎢0  0  0  0  0  0  0  0  0    1          0           0    ⎥
⎢                                                         ⎥
⎢0  0 

In [322]:
print(f"final_b_matrix (shape:({final_b_matrix.shape}))")
pprint(final_b_matrix)


final_b_matrix (shape:((12, 4)))
⎡                       0                                                0    
⎢                                                                             
⎢                       0                                                0    
⎢                                                                             
⎢                       0                                                0    
⎢                                                                             
⎢2⋅kf⋅u₁⋅(sin(φ)⋅sin(ψ)⋅cos(θ) + sin(θ)⋅cos(ψ))   2⋅kf⋅u₂⋅(sin(φ)⋅sin(ψ)⋅cos(θ
⎢──────────────────────────────────────────────   ────────────────────────────
⎢                      m                                                m     
⎢                                                                             
⎢2⋅kf⋅u₁⋅(-sin(φ)⋅cos(ψ)⋅cos(θ) + sin(ψ)⋅sin(θ))  2⋅kf⋅u₂⋅(-sin(φ)⋅cos(ψ)⋅cos(
⎢───────────────────────────────────────────────  ────────────────────────────
⎢                  

In [323]:


values = {
    L: 0.0397,
    I1: 2.3951e-5,
    I2: 2.3951e-5,
    I3: 3.2347e-5,
    px: 10, py: 10, pz: 10,
    vx: 0, vy: 0, vz: 0,
    ax: 0, ay: 0, az: 0,
    # x: 0, v: 0, u: 0,
    p: 0., q: 0., r: 0.,
    phi: 0.0, theta: 0.0, psi: 0.0,
    g: -9.81,
    km: 7.94e-12,
    kf: 3.16e-10,
    u1: 25.0, u2: 26.0, u3: 27.0, u4: 28.0,
    # u1: 0, u2: 0, u3: 0, u4: 0,
    F1: u1**2 *kf,
    F2: u2**2 *kf,
    F3: u3**2 *kf,
    F4: u4**2 *kf,
    m: 0.027
}

# Substitute values into the matrix
final_a_matrix_numeric = final_a_matrix.subs(values)
final_b_matrix_numeric = final_b_matrix.subs(values)


In [324]:
print(f"final_a_matrix with substituted values: ")
pprint(final_a_matrix_numeric)

final_a_matrix with substituted values: 
⎡0  0  0  1  0  0  0  0  0  0  0  0⎤
⎢                                  ⎥
⎢0  0  0  0  1  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  1  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  1  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  1  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  1⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎢0  0  0  0  0  0  0  0  0  0  0  0⎥
⎢                                  ⎥
⎣0  0  0  0  0  0  0  0  0  0  0  0⎦


In [325]:
print(f"final_b_matrix with substituted values: ")
pprint(final_b_matrix_numeric)

final_b_matrix with substituted values: 
⎡         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢         0                     0                    0                    0   
⎢                                                                             
⎢5.85185185185185e-7   6.08592592592593e-7        6.32e-7        6.55407407407
⎢                                                                             
⎢         0

In [326]:
# Convert to a NumPy array
final_a_matrix_numpy = np.array(final_a_matrix_numeric.evalf(), dtype=np.float32)
final_b_matrix_numpy = np.array(final_b_matrix_numeric.evalf(), dtype=np.float32)


In [327]:

print(f"final_a_matrix (shape : {final_a_matrix_numeric.shape})")
pprint(final_a_matrix_numpy)


final_a_matrix (shape : (12, 12))
 [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [328]:
print(f"final_b_matrix (shape : {final_b_matrix_numeric.shape})")
pprint(final_b_matrix_numpy)

final_b_matrix (shape : (12, 4))
 [[ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 5.8518521e-07  6.0859259e-07  6.3200002e-07  6.5540740e-07]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00]
  [ 0.0000000e+00  2.7236876e-05  0.0000000e+00 -2.9332019e-05]
  [-2.6189304e-05  0.0000000e+00  2.8284447e-05  0.0000000e+00]
 [ 1.2273163e-05 -1.2764090e-05  1.3255016e-05 -1.3745943e-05]]


In [329]:
def get_controllability_matrix(a, b):
    n = a.shape[1]
    m = b.shape[1]
    ctrl_matrix = b
    for i in range(1, n):
        ctrl_matrix = np.hstack((ctrl_matrix, np.linalg.matrix_power(a, i)@b))
    assert ctrl_matrix.shape == (n, n*m), f"Controllability matrix does not have the proper shape of ({(n, n*m)})"
    return ctrl_matrix

def is_controllable(c):
    rank_C = np.linalg.matrix_rank(c)
    return rank_C

In [335]:
def get_controllable_subsystem(a, b, c, rank_C):
    """
    a := state dynamics w.r.t. current state
    b := state dynamics w.r.t. input
    c := controllability matrix
    """
    # Use QR decomposition to find an orthonormal basis for the controllable subspace
    Q, _ = np.linalg.qr(c)  # Q has orthonormal columns spanning the controllable subspace
    T = Q  # Transformation matrix

    # Transform A and B
    A_transformed = np.linalg.inv(T) @ a @ T
    B_transformed = np.linalg.inv(T) @ b
    print(f"New state system: ")
    print(f"Transformed A: \n{A_transformed}")
    print(f"Transformed B: \n{B_transformed}")
    # Identify which states are controllable
    controllable_states = np.where(np.sum(np.abs(T), axis=1) > 1e-6)[0]
    uncontrollable_states = np.setdiff1d(np.arange(A.shape[0]), controllable_states)
    print(f"Controllable states: {controllable_states}")
    print(f"Uncontrollable states: {uncontrollable_states}")

    # Extract controllable part
    A_c = A_transformed[:rank_C, :rank_C]
    B_c = B_transformed[:rank_C, :]

    return A_c, B_c, T

In [336]:
A_prev = final_a_matrix_numpy
# Adjust this value if needed (try 1e-5, 1e-7, etc.)
A = A_prev
B = final_b_matrix_numpy
Q = np.eye(12)
R = np.eye(4)
print("Condition number of A:", np.linalg.cond(A))
print("Condition number of B:", np.linalg.cond(B))
ctrl_matrix = get_controllability_matrix(A, B)
ctrl_rank = is_controllable(ctrl_matrix)
can_ctrl = ctrl_rank >= A.shape[1]
print(f"Is this linear system controllable? {can_ctrl}")
print(f"Controllability matrix has a rank of {ctrl_rank} for a system with {A.shape[1]} state dimensions.")

# get the controllable subsystem:
A_c, B_c, T = get_controllable_subsystem(A, B, ctrl_matrix, ctrl_rank)


Condition number of A: 20000000001.934486
Condition number of B: 32.38606
Is this linear system controllable? False
Controllability matrix has a rank of 7 for a system with 12 state dimensions.
New state system: 
Transformed A: 
[[ 1.00000000e-05  2.89708135e-22 -4.27732547e-22  6.18327868e-23
   9.00812436e-35 -1.06842091e-32 -1.01332683e-23  2.77733908e-17
   3.84006574e-17 -2.56351514e-16  2.54639534e-18 -8.08990172e-25]
 [ 4.30765336e-22  1.00000000e-05  5.20156045e-22 -4.49929804e-23
   4.32312268e-35 -2.11355493e-32 -9.25923352e-23  2.53778256e-16
   1.52372883e-17 -1.22637449e-16  1.45109781e-18 -4.21782684e-25]
 [ 2.54295445e-18  9.67399799e-18  1.00000000e-05 -3.60191432e-17
   1.89148762e-33 -1.83606717e-32 -3.10378995e-23  8.50690710e-17
   4.97337823e-17 -3.36238853e-16  3.38704152e-18 -1.06812886e-24]
 [ 2.16151262e-17  2.53147385e-17  6.07057343e-17  1.00000000e-05
  -5.31501395e-32 -3.57897125e-31 -1.33280264e-21  3.65296251e-15
   6.12779829e-16 -4.36731522e-15  4.64610

In [332]:
print(f"Transformation matrix T (shape ({T.shape})): \n{T}")

Transformation matrix T (shape ((12, 12))): 
[[ 0.00000000e+00  8.32667268e-17  2.77555756e-16  4.41579597e-15
  -7.12002768e-18  4.82847379e-19  4.68546498e-09 -1.28419822e-02
  -9.71877835e-01 -1.45137977e-01  1.84996013e-01  8.48922871e-09]
 [-0.00000000e+00  0.00000000e+00 -6.07153217e-17 -3.63944985e-15
   3.83329793e-18 -2.21327132e-18  5.15545282e-10 -1.41301300e-03
  -1.81494879e-01 -3.71017350e-02 -9.82690731e-01  1.81811445e-07]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00 -1.06165077e-15
  -4.76259607e-02  3.15124338e-02  9.98368036e-01  3.64259843e-07
   1.66533454e-16 -1.05297715e-15 -5.55111512e-17  1.32346251e-17]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   3.47395503e-19 -3.43075914e-17 -1.30029709e-09  3.56386880e-03
   1.49455544e-01 -9.88714238e-01  9.72082223e-03 -3.10519945e-09]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  6.65700134e-17  3.64822506e-07 -9.99910189e-01
   1.32711244e-02 -1.607507

In [333]:
print(f"A_c (shape ({A_c.shape})): \n{A_c}")
print(f"B_c (shape ({B_c.shape})): \n{B_c}")


A_c (shape ((7, 7))): 
[[ 1.00000000e-05  2.89708135e-22 -4.27732547e-22  6.18327868e-23
   9.00812436e-35 -1.06842091e-32 -1.01332683e-23]
 [ 4.30765336e-22  1.00000000e-05  5.20156045e-22 -4.49929804e-23
   4.32312268e-35 -2.11355493e-32 -9.25923352e-23]
 [ 2.54295445e-18  9.67399799e-18  1.00000000e-05 -3.60191432e-17
   1.89148762e-33 -1.83606717e-32 -3.10378995e-23]
 [ 2.16151262e-17  2.53147385e-17  6.07057343e-17  1.00000000e-05
  -5.31501395e-32 -3.57897125e-31 -1.33280264e-21]
 [ 4.24741900e-01 -3.52263933e-01  8.33969088e-01  2.40601379e-10
   1.00000000e-05  7.54700024e-18 -2.97862001e-17]
 [-7.84865292e-20 -1.30211537e+00 -5.50006334e-01  3.15168493e-02
   1.68950127e-18  1.00000000e-05  5.33033327e-17]
 [-2.68914034e-18 -1.75669628e-18  2.88645107e-10 -1.00049752e+00
   5.09153281e-17 -3.32251289e-17  1.00000000e-05]]
B_c (shape ((7, 4))): 
[[-2.89284046e-05  5.40298059e-06  1.99699628e-05  5.81859454e-06]
 [-1.06478264e-21 -2.95964118e-05  9.34913674e-06  2.21140658e-05]


In [334]:
Q_c = np.eye(ctrl_rank)
R_c = np.eye(B_c.shape[1])
S = scipy.linalg.solve_continuous_are(A_c, B_c, Q_c, R_c)
K = -np.linalg.solve(R_c, B_c.T @ S)
print(f"final_a_matrix (shape : {final_a_matrix_numeric.shape})\n{final_a_matrix_numeric}")
print(f"K (shape : {K.shape})\n{K}")





final_a_matrix (shape : (12, 12))
Matrix([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
K (shape : (4, 7))
[[ 6.97597458e+01 -3.41187731e+01  1.44572143e+02  6.22765972e+02
   4.96985390e-01 -7.46010825e-03 -4.71147608e-01]
 [-4.79357169e+01  2.38709873e+02 -1.00833763e+01  6.52135576e+02
  -4.91121730e-01 -7.07768788e-01 -5.17530956e-01]
 [ 7.34846118e+01 -3.71870377e+01  1.56566442e+02  6.69331604e+02
   5.36735182e-01 -8.05397639e-03 -5.08746161e-01]
 [-4.55958611e+01 -1.11848758e+02 -1.52743475e+02  6.22132050e+02
  -4.78884696e-01  7.09037249e-01 -5.26327665e-01]]


In [342]:
from control import ctrb; 
# C = ctrb(A,B); 
# Q, _ = np.linalg.qr(C); 
# T = np.hstack((Q[:,:7], scipy.linalg.null_space(C.T)))
# print(T)
A = np.eye(12)
C = ctrb(A, B); 
rank = np.linalg.matrix_rank(C)
print(rank)

C = ctrb(A, B)  # 12 × 48

# QR decomposition
Q, R = np.linalg.qr(C)
Q_c = Q[:, :7]  # 12 × 7, controllable subspace

# Print Q_c to see significant entries
print(Q_c)

4
[[ 0.00000000e+00  8.32667268e-17  2.77555756e-16  4.41579597e-15
  -1.00000000e+00  3.10838398e-18  1.84634873e-17]
 [-0.00000000e+00  0.00000000e+00 -6.07153217e-17 -3.63944985e-15
  -7.69783542e-18  1.00000000e+00 -7.36222816e-17]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00 -1.06165077e-15
   7.37257477e-18  0.00000000e+00  1.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.78141682e-18 -1.38043730e-16  3.14182829e-17]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  2.77555756e-17  5.55111512e-17]
 [-2.02287410e-02 -2.42559161e-02 -5.70506170e-02 -9.97871573e-01
  -4.43048376e-15 -3.64632587e-15 -1.07000194e-15]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e

# Next Steps

1) We want to be able to concretely say which states are controllable. It does not matter that 7 states are controllable we cannot control for example, the altitude of the quadrotor. We need to answer this question.
2) $K$ is our linear state-feedback controller. Our controller can be written as $u = -Kx$. In other words, based on our current state $x$, $u$ will give us the control input we should use to move towards equilibrium. **However**, we only have $K$ for the 7 state sub-system. Can we write what our control would look like for our original 12 state-system? Is it necessary to rewrite K? 
3) Once we know which states are controllable, and we are satisfied with our controller, we should create some plot or visualization to quickly confirm that this controller can stabilize our system for points *near* the equilibrium point. 